# Validation

## 0. Merging the dataset

In [2]:
def load_jsonl_as_dict(path, key):
    with open(path, "r", encoding="utf-8") as f:
        return {json.loads(line)[key]: json.loads(line) for line in f}

scores = load_jsonl("scores.jsonl")
articles = load_jsonl_as_dict("articles.jsonl", "article_id")
summaries = load_jsonl_as_dict("summaries.jsonl", "article_id")

# Merge
merged = []
for s in scores:
    aid = s["article_id"]
    
    article = articles.get(aid, {})
    summary = summaries.get(aid, {})
    
    merged.append({
        **s,
        "article": article.get("article", ""),
        "summary": summary.get("summary", "")
    })

FileNotFoundError: [Errno 2] No such file or directory: 'articles.jsonl'

## 1. Manual inspection

Let's observe 5 high scores and 5 low score. Of course, we will translate if as before:

In [1]:
from translatepy import Translator
import json

# ----------------------------
# Load results
# ----------------------------
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

data = load_jsonl("scores.jsonl")

# ----------------------------
# Sort by global score
# ----------------------------
sorted_data = sorted(data, key=lambda x: x["global_score"], reverse=True)

top_5 = sorted_data[:5]
bottom_5 = sorted_data[-5:]

translator = Translator()

# ----------------------------
# Helper print function
# ----------------------------
def show_examples(examples, title):
    print("\n" + "="*50)
    print(title)
    print("="*50)

    for i, ex in enumerate(examples):
        print(ex)
        print(f"\n--- Example {i+1} ---")
        print(f"Summary ID: {ex['summary_id']}")
        print(f"Global score: {ex['global_score']:.3f}")
        print(f"Label: {ex['label']}")

        translated = translator.translate(ex["summary"], "French")
        print("\nOriginal:")
        print(ex["summary"])

        print("\nTranslated (FR):")
        print(translated)

# ----------------------------
# Display
# ----------------------------
show_examples(top_5, "TOP 5 HIGHEST SCORES")
show_examples(bottom_5, "BOTTOM 5 LOWEST SCORES")


TOP 5 HIGHEST SCORES
{'summary_id': '45715110_979831e4', 'article_id': '45715110', 'relevance': 0.9987775087356567, 'hallucination': 8.571428433690187e-09, 'salience': 0.17777777777777778, 'compression_ratio': 0.1052104207362621, 'compression_score': 1.0, 'truncation': 1.0, 'noise': False, 'global_score': 0.7353721792944755, 'label': 'good'}

--- Example 1 ---
Summary ID: 45715110_979831e4
Global score: 0.735
Label: good


KeyError: 'summary'

In [ ]:
✔️ B. Sanity checks

Exemples :

duplicates → high duplication score ✔️
hallucinated summaries → low faithfulness ✔️
truncated → low truncation score ✔️


✔️ C. Correlation quick check (bonus fort)

Même simple :

relevance vs salience
hallucination vs similarity


✔️ D. Distribution plots (très bien vu)
histogram global scores
histogram compression
histogram similarity